# Kaizen Knowledge DB — inspect & manage the 3 ChromaDB collections

One persistent ChromaDB lives at `data/chroma_db/chroma.sqlite3` and holds **3 collections**:

| # | collection name | what it holds | built from |
|---|-----------------|---------------|------------|
| 1 | `glayout_knowledge` | DRC-clean glayout layout code | `data/glayout_code/*.jsonl` + `data/circuits/*/*_clean.py` |
| 2 | `rf_theory` | RF/EM/analog theory, EE-QA, PySpice | `data/rf_theory/**` |
| 3 | `error_feedback` | error -> root cause -> fix memory | empty at build; the Kaizen loop writes to it at runtime |

This notebook shows how to **see what is inside** and **manage (add / delete / reset / rebuild)** each one.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath('../src'))
os.environ.setdefault('PDK_ROOT', os.path.expanduser('~/pdks'))
from gelochip.kaizen import collections, config

print('ChromaDB dir :', config.CHROMA_DIR)
print('collections  :', config.ALL_COLLECTIONS)
print('embed model  :', config.EMBED_MODEL)
print()
for name, n in collections.collection_counts().items():
    print(f'  {name:20s} {n:6d} chunks')

ChromaDB dir : /home/irman/Gelochip/data/chroma_db
collections  : ('glayout_knowledge', 'rf_theory', 'error_feedback')
embed model  : /home/irman/Gelochip/models/embeddings/all-MiniLM-L6-v2



Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  glayout_knowledge       216 chunks
  rf_theory              6231 chunks
  error_feedback            0 chunks


## 1. Peek inside a collection — documents + metadata

`get_vectorstore(name)._collection` is the raw chromadb Collection. Use `.get(...)` to read rows,
`.peek(n)` for a quick sample. `include=` controls which fields come back.

In [3]:
def peek(name, n=5, where=None):
    """Print the first n chunks of a collection (optionally filtered by metadata)."""
    col = collections.get_vectorstore(name)._collection
    got = col.get(include=['documents', 'metadatas'], limit=n, where=where)
    print(f'### {name}: showing {len(got["ids"])} of {col.count()} chunks',
          f'(filter={where})' if where else '')
    for i, (cid, doc, meta) in enumerate(zip(got['ids'], got['documents'], got['metadatas'])):
        print(f'\n--- [{i}] id={cid}  meta={meta}')
        print(doc[:400].replace('\n', ' '))

peek('rf_theory', 3)

### rf_theory: showing 3 of 6231 chunks 

--- [0] id=theory-bowick-rf-circuit-design-00000  meta={'source': 'bowick_rf_circuit_design_2e', 'doc_type': 'book'}
--- PAGE BREAK ---  RF CIRCUIT DESIGN  --- PAGE BREAK ---  This page intentionally left blank   --- PAGE BREAK ---  RF CIRCUIT DESIGN CHRISTOPHER BOWICK WITH JOHN BLYLER AND CHERYL AJLUNI AMSTERDAM • BOSTON • HEIDELBERG • LONDON • NEW YORK • OXFORD PARIS • SAN DIEGO • SAN FRANCISCO • SINGAPORE • SYDNEY • TOKYO Newnes is an imprint of Elsevier  --- PAGE BREAK ---

--- [1] id=theory-bowick-rf-circuit-design-00001  meta={'source': 'bowick_rf_circuit_design_2e', 'doc_type': 'book'}
Cover image by iStockphoto Newnes is an imprint of Elsevier 30 Corporate Drive, Suite 400, Burlington, MA 01803, USA Linacre House, Jordan Hill, Oxford OX2 8DP, UK Copyright © 2008, Elsevier Inc. All rights reserved. No part of this publication may be reproduced, stored in a retrieval system, or transmitted in any form or by any means, electronic, mechanic

In [4]:
# rf_theory — and a metadata filter example (only the PySpice analog examples)
peek('rf_theory', 2)
peek('rf_theory', 2, where={'doc_type': 'pyspice_example'})

### rf_theory: showing 2 of 6231 chunks 

--- [0] id=theory-bowick-rf-circuit-design-00000  meta={'doc_type': 'book', 'source': 'bowick_rf_circuit_design_2e'}
--- PAGE BREAK ---  RF CIRCUIT DESIGN  --- PAGE BREAK ---  This page intentionally left blank   --- PAGE BREAK ---  RF CIRCUIT DESIGN CHRISTOPHER BOWICK WITH JOHN BLYLER AND CHERYL AJLUNI AMSTERDAM • BOSTON • HEIDELBERG • LONDON • NEW YORK • OXFORD PARIS • SAN DIEGO • SAN FRANCISCO • SINGAPORE • SYDNEY • TOKYO Newnes is an imprint of Elsevier  --- PAGE BREAK ---

--- [1] id=theory-bowick-rf-circuit-design-00001  meta={'doc_type': 'book', 'source': 'bowick_rf_circuit_design_2e'}
Cover image by iStockphoto Newnes is an imprint of Elsevier 30 Corporate Drive, Suite 400, Burlington, MA 01803, USA Linacre House, Jordan Hill, Oxford OX2 8DP, UK Copyright © 2008, Elsevier Inc. All rights reserved. No part of this publication may be reproduced, stored in a retrieval system, or transmitted in any form or by any means, electronic, mechanic

## 2. What sources / doc_types are in a collection (composition audit)

This is the most useful health check: it shows duplicates, junk extractions (sources with only 1 chunk),
and the relevance mix.

In [5]:
from collections import Counter

def audit(name):
    col = collections.get_vectorstore(name)._collection
    n = col.count()
    got = col.get(include=['metadatas'], limit=n)
    metas = got['metadatas']
    print(f'### {name}: {n} chunks')
    print('  by doc_type:', dict(Counter(m.get('doc_type', '?') for m in metas)))
    print('  by source (top 20):')
    for s, c in Counter(m.get('source', '?') for m in metas).most_common(20):
        flag = '  <-- only 1 chunk (likely junk / failed extraction)' if c == 1 else ''
        print(f'    {c:6d}  {s}{flag}')

for nm in config.ALL_COLLECTIONS:
    audit(nm); print()

### glayout_knowledge: 216 chunks
  by doc_type: {'code_template': 201, 'clean_circuit': 15}
  by source (top 20):
       155  dataset_full.jsonl
        31  dataset_circuits.jsonl
        15  dataset_clean_circuits.jsonl
         1  current_mirror_clean.py  <-- only 1 chunk (likely junk / failed extraction)
         1  diff_pair_clean.py  <-- only 1 chunk (likely junk / failed extraction)
         1  diff_pair_cmirrorbias_clean.py  <-- only 1 chunk (likely junk / failed extraction)
         1  diff_pair_stackedcmirror_clean.py  <-- only 1 chunk (likely junk / failed extraction)
         1  differential_to_single_ended_converter_clean.py  <-- only 1 chunk (likely junk / failed extraction)
         1  fvf_clean.py  <-- only 1 chunk (likely junk / failed extraction)
         1  low_voltage_cmirror_clean.py  <-- only 1 chunk (likely junk / failed extraction)
         1  n_block_clean.py  <-- only 1 chunk (likely junk / failed extraction)
         1  opamp_clean.py  <-- only 1 chunk (likel

## 3. Semantic search (what the agent actually retrieves)

This is exactly how the Kaizen agent queries each collection during generation.

In [6]:
def search(name, query, k=3, where=None):
    store = collections.get_vectorstore(name)
    docs = store.similarity_search(query, k=k, filter=where)
    print(f'### {name} <- "{query}"  ({len(docs)} hits)')
    for d in docs:
        print(f'\n--- source={d.metadata.get("source")} circuit={d.metadata.get("circuit")}')
        print(d.page_content[:300].replace('\n', ' '))

search('glayout_knowledge', 'differential pair with current mirror load on gf180')
search('rf_theory', 'common source amplifier gain and output resistance')

### glayout_knowledge <- "differential pair with current mirror load on gf180"  (3 hits)

--- source=dataset_circuits.jsonl circuit=current_mirror
# Task What is a differential pair with stacked current mirror and how is it implemented in glayout for gf180 PDK?  # glayout solution A differential pair with stacked current mirror is A differential pair with a stacked current mirror load, providing high output impedance.  Here is the glayout Pyth

--- source=dataset_circuits.jsonl circuit=current_mirror
# Task What is a differential pair with current mirror bias and how is it implemented in glayout for gf180 PDK?  # glayout solution A differential pair with current mirror bias is A differential pair biased by a current mirror.  Here is the glayout Python implementation:  ```python import sys try:  

--- source=dataset_full.jsonl circuit=current_mirror
# Task <image> This is the current GDS layout of a differential pair with stacked current mirror circuit. Write the glayout Python code to 

## 4. Manage data — delete specific rows or by filter

**Delete by id** (e.g. a junk chunk you spotted in `peek`) or **delete by metadata filter**
(e.g. remove the broken login-page extractions). These edit the live DB immediately.

In [ ]:
def delete_ids(name, ids):
    col = collections.get_vectorstore(name)._collection
    col.delete(ids=ids); print(f'{name}: deleted {len(ids)} ids -> now {col.count()} chunks')

def delete_where(name, where):
    """Delete every chunk matching a metadata filter, e.g. {'source': 'steer_v2_transmission_lines'}."""
    col = collections.get_vectorstore(name)._collection
    before = col.count(); col.delete(where=where)
    print(f'{name}: deleted where {where} -> {before} -> {col.count()} chunks')

# EXAMPLE (commented out — uncomment to run):
# remove the 3 broken steer volumes that are just NC-State login pages:
# for src in ('steer_v1_radio_systems', 'steer_v2_transmission_lines', 'steer_v3_networks'):
#     delete_where('rf_theory', {'source': src})

## 5. Manage data — add your own knowledge

Use the typed helpers so metadata + (optionally) the JSONL dataset stay in sync.

In [ ]:
# Add a verified glayout snippet to collection 1 (also appends to a JSONL dataset by default):
# collections.add_template(
#     instruction='Generate a DRC-clean current mirror on gf180',
#     code='from glayout... \ncomponent = top',
#     circuit='current_mirror', source='corrected', also_jsonl=True)

# Add an error->fix lesson to collection 3 (this is what the agent does after a failed attempt):
# collections.add_lesson(
#     scenario='met2 routes too close on gf180',
#     error='M2.2a Metal2 spacing < 0.28um',
#     root_cause='util_max_metal_seperation defaulted to sky130 0.3um',
#     fix='raise GF180_MIN_METAL_SEP to 0.5 and re-route')
print('add_template / add_lesson ready — uncomment to use')

## 6. Reset / rebuild a collection

- `_reset(store)` empties one collection (keeps it registered).
- `ingest_templates` / `ingest_theory` / `ingest_lessons` rebuild from source.
- `build_all` rebuilds everything (collection 3 always starts empty unless `seed_feedback=True`).

**Use `parse_pdfs=True`** to also OCR/parse the raw PDFs in `data/rf_theory/books` and `.../arxiv/pdfs`
(by default only the pre-extracted `texts/*.txt` + abstracts are ingested, so the raw PDFs contribute nothing).

In [ ]:
# --- empty just the error_feedback collection (keeps it, agent keeps using it) ---
# collections._reset(collections.get_vectorstore(config.COLL_LESSONS))

# --- rebuild collection 1 from the (corrected) clean.py + jsonl ---
# print('glayout_knowledge ->', collections.ingest_templates(reset=True), 'chunks')

# --- rebuild collection 2; parse_pdfs=True also ingests the raw PDFs ---
# print('rf_theory ->', collections.ingest_theory(reset=True, parse_pdfs=False), 'chunks')

# --- rebuild everything at once ---
# print(collections.build_all(parse_pdfs=False, seed_feedback=False))
print('rebuild helpers ready — uncomment the one you want')

## 7. Raw SQLite view (no embeddings model needed)

If you just want to poke the file directly, ChromaDB is a normal SQLite DB. The collection
registry and per-chunk metadata are plain tables.

In [ ]:
import sqlite3
db = sqlite3.connect(str(config.CHROMA_DIR / 'chroma.sqlite3'))
cur = db.cursor()
print('tables:', [r[0] for r in cur.execute(
    "select name from sqlite_master where type='table' order by name")])
print('\ncollections registered:')
for row in cur.execute('select name, id from collections'):
    print('  ', row)
print('\nrows in embedding_metadata (sample of distinct metadata keys):')
for row in cur.execute('select distinct key from embedding_metadata limit 20'):
    print('  ', row[0])
db.close()

## Cheat-sheet — managing each of the 3 knowledges

**1. `glayout_knowledge`** (layout code the agent imitates)
- Source of truth = `data/circuits/*/*_clean.py` (verified) + `data/glayout_code/*.jsonl`.
- To update: fix the clean.py / jsonl, then `ingest_templates(reset=True)`.
- Add one snippet on the fly: `add_template(instruction, code, circuit=...)`.

**2. `rf_theory`** (background theory)
- Source = `data/rf_theory/texts/*.txt`, `arxiv/all_abstracts.txt`, `huggingface/*.jsonl`, `analog_pyspice/sft_pairs.jsonl`.
- Raw PDFs in `books/` and `arxiv/pdfs/` are **only** ingested when `ingest_theory(parse_pdfs=True)`.
- Audit with cell 2; delete junk sources with `delete_where('rf_theory', {'source': '...'})`.

**3. `error_feedback`** (runtime memory)
- Starts empty; the Kaizen loop appends via `add_lesson(...)` after failures.
- Empty it (keep using it) with `_reset(get_vectorstore(config.COLL_LESSONS))`.